<a href="https://colab.research.google.com/github/JDaviA/Desafio/blob/main/Recomenda%C3%A7%C3%A3o_por_Imagens_Digitais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')

# Caminho para a pasta de imagens no Google Drive e na máquina local
IMAGE_FOLDER_DRIVE = '/content/drive/MyDrive/imagens_produtos/'
IMAGE_FOLDER_LOCAL = './imagens_produtos/'

# Carregar modelo pré-treinado ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
model = Model(inputs=base_model.input, outputs=base_model.output)

def extract_features(img_path):
    """Extrai características da imagem usando ResNet50."""
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    features = model.predict(img_array)
    return features.flatten()

# Carregar imagens e extrair características
image_features = {}
image_paths = []

# Adicionar imagens Google Drive
if os.path.exists(IMAGE_FOLDER_DRIVE):
    image_paths += [os.path.join(IMAGE_FOLDER_DRIVE, f) for f in os.listdir(IMAGE_FOLDER_DRIVE) if f.endswith(('jpg', 'png', 'jpeg'))]

# Adicionar imagens da máquina local
if os.path.exists(IMAGE_FOLDER_LOCAL):
    image_paths += [os.path.join(IMAGE_FOLDER_LOCAL, f) for f in os.listdir(IMAGE_FOLDER_LOCAL) if f.endswith(('jpg', 'png', 'jpeg'))]

for img_path in image_paths:
    image_features[img_path] = extract_features(img_path)

def find_similar_images(query_img_path, top_n=5):
    """Encontra imagens mais semelhantes com base na similaridade do cosseno."""
    query_features = extract_features(query_img_path)
    similarities = {}

    for img_path, features in image_features.items():
        sim = cosine_similarity([query_features], [features])[0][0]
        similarities[img_path] = sim

    # Ordenar pelas imagens similares
    sorted_images = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return sorted_images

def show_images(query_img_path, similar_images):
    """Mostra a imagem de consulta e as recomendadas."""
    fig, axes = plt.subplots(1, len(similar_images) + 1, figsize=(15, 5))

    query_img = Image.open(query_img_path)
    axes[0].imshow(query_img)
    axes[0].set_title("Consulta")
    axes[0].axis("off")

    # Mostrar imagens recomendadas
    for i, (img_path, sim) in enumerate(similar_images):
        img = Image.open(img_path)
        axes[i+1].imshow(img)
        axes[i+1].set_title(f"Sim: {sim:.2f}")
        axes[i+1].axis("off")

    plt.show()

# Substitua pelo caminho da imagem de teste)
query_image_path = './imagens_produtos/teste.jpg'  # Ou '/content/drive/MyDrive/imagens_produtos/teste.jpg'
similar_images = find_similar_images(query_image_path)
show_images(query_image_path, similar_images)
